# ETL 04 - Transform + Load: TLC 2026 a Parquet y R2

Objetivo: transformar los 4 tipos de records a una tabla analítica común, escribir un Parquet particionado localmente y subirlo a Cloudflare R2.

## Dependencias

```bash
conda install pandas pyarrow s3fs python-dotenv
```

In [3]:
from pathlib import Path
import os

import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import s3fs

PROJECT_DIR = Path.cwd()
BASE_DIR = PROJECT_DIR if PROJECT_DIR.name == "etl-04" else PROJECT_DIR / "etl-04"
BRONZE_DIR = BASE_DIR / "data" / "bronze" / "tlc"
SILVER_DIR = BASE_DIR / "data" / "silver" / "tlc"
OUTPUT_DIR = SILVER_DIR / "tlc_2026_clean"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

RECORD_TYPES = {"yellow", "green", "fhv", "fhvhv"}


def parse_tlc_filename(filename: str) -> dict | None:
    """Extrae tipo, año y mes desde nombres como yellow_tripdata_2026-01.parquet."""
    if not filename.endswith(".parquet") or "_tripdata_" not in filename:
        return None

    record_type, year_month_ext = filename.split("_tripdata_", 1)
    year_month = year_month_ext.removesuffix(".parquet")

    if record_type not in RECORD_TYPES or "-" not in year_month:
        return None

    year_text, month_text = year_month.split("-", 1)
    if not year_text.isdigit() or not month_text.isdigit():
        return None

    return {
        "record_type": record_type,
        "year": int(year_text),
        "month": int(month_text),
        "year_month": year_month,
    }


## 1. Transformaciones comunes

La funcion toma un archivo original y devuelve un DataFrame con columnas estandarizadas. Algunas columnas no existen en todos los tipos; quedan como `NA` para mantener una salida rectangular.

In [4]:
COLUMN_MAP = {
    "yellow": {
        "pickup_datetime": "tpep_pickup_datetime",
        "dropoff_datetime": "tpep_dropoff_datetime",
        "passenger_count": "passenger_count",
        "trip_distance": "trip_distance",
        "pickup_location_id": "PULocationID",
        "dropoff_location_id": "DOLocationID",
        "payment_type": "payment_type",
        "fare_amount": "fare_amount",
        "tip_amount": "tip_amount",
        "total_amount": "total_amount",
        "cbd_congestion_fee": "cbd_congestion_fee",
    },
    "green": {
        "pickup_datetime": "lpep_pickup_datetime",
        "dropoff_datetime": "lpep_dropoff_datetime",
        "passenger_count": "passenger_count",
        "trip_distance": "trip_distance",
        "pickup_location_id": "PULocationID",
        "dropoff_location_id": "DOLocationID",
        "payment_type": "payment_type",
        "fare_amount": "fare_amount",
        "tip_amount": "tip_amount",
        "total_amount": "total_amount",
        "cbd_congestion_fee": "cbd_congestion_fee",
    },
    "fhv": {
        "pickup_datetime": "pickup_datetime",
        "dropoff_datetime": "dropOff_datetime",
        "pickup_location_id": "PUlocationID",
        "dropoff_location_id": "DOlocationID",
        "dispatching_base_num": "dispatching_base_num",
        "affiliated_base_number": "Affiliated_base_number",
    },
    "fhvhv": {
        "pickup_datetime": "pickup_datetime",
        "dropoff_datetime": "dropoff_datetime",
        "pickup_location_id": "PULocationID",
        "dropoff_location_id": "DOLocationID",
        "dispatching_base_num": "dispatching_base_num",
        "originating_base_num": "originating_base_num",
        "trip_miles": "trip_miles",
        "trip_time": "trip_time",
        "base_passenger_fare": "base_passenger_fare",
        "tips": "tips",
        "driver_pay": "driver_pay",
        "shared_request_flag": "shared_request_flag",
        "shared_match_flag": "shared_match_flag",
        "cbd_congestion_fee": "cbd_congestion_fee",
    },
}

OUTPUT_COLUMNS = [
    "record_type",
    "year_month",
    "source_file",
    "pickup_datetime",
    "dropoff_datetime",
    "duration_min",
    "pickup_location_id",
    "dropoff_location_id",
    "passenger_count",
    "trip_distance",
    "trip_miles",
    "trip_time",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "base_passenger_fare",
    "tips",
    "driver_pay",
    "cbd_congestion_fee",
    "dispatching_base_num",
    "originating_base_num",
    "affiliated_base_number",
    "shared_request_flag",
    "shared_match_flag",
]


In [6]:
def read_parquet_in_batches(path: Path, batch_size: int = 250_000):
    parquet_file = pq.ParquetFile(path)
    for batch in parquet_file.iter_batches(batch_size=batch_size):
        yield batch.to_pandas()


def transform_batch(df: pd.DataFrame, record_type: str, year_month: str, source_file: str) -> pd.DataFrame:
    mapping = COLUMN_MAP[record_type]
    out = pd.DataFrame(index=df.index)
    out["record_type"] = record_type
    out["year_month"] = year_month
    out["source_file"] = source_file

    for output_col in OUTPUT_COLUMNS:
        if output_col in {"record_type", "year_month", "source_file", "duration_min"}:
            continue
        source_col = mapping.get(output_col)
        out[output_col] = df[source_col] if source_col in df.columns else pd.NA

    out["pickup_datetime"] = pd.to_datetime(out["pickup_datetime"], errors="coerce")
    out["dropoff_datetime"] = pd.to_datetime(out["dropoff_datetime"], errors="coerce")
    out["duration_min"] = (out["dropoff_datetime"] - out["pickup_datetime"]).dt.total_seconds() / 60

    numeric_cols = [
        "duration_min",
        "passenger_count",
        "trip_distance",
        "trip_miles",
        "trip_time",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "base_passenger_fare",
        "tips",
        "driver_pay",
        "cbd_congestion_fee",
    ]
    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")

    for col in ["pickup_location_id", "dropoff_location_id", "payment_type"]:
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")

    text_cols = [
        "record_type",
        "year_month",
        "source_file",
        "dispatching_base_num",
        "originating_base_num",
        "affiliated_base_number",
        "shared_request_flag",
        "shared_match_flag",
    ]
    for col in text_cols:
        out[col] = out[col].astype("string")

    valid = out["pickup_datetime"].notna() & out["dropoff_datetime"].notna()
    valid &= out["duration_min"].between(0, 24 * 60, inclusive="both")
    valid &= out["pickup_datetime"].dt.strftime("%Y-%m").eq(year_month)

    if record_type in {"yellow", "green"}:
        valid &= out["trip_distance"].ge(0) | out["trip_distance"].isna()
        valid &= out["total_amount"].ge(0) | out["total_amount"].isna()

    return out.loc[valid, OUTPUT_COLUMNS].reset_index(drop=True)


def to_arrow(df: pd.DataFrame) -> pa.Table:
    """Pasa a Arrow dejando todo el texto como `string`.

    Segun la version de pandas/pyarrow el texto sale como `string` o `large_string`;
    fijarlo evita que los archivos de silver queden con tipos distintos entre si.
    """
    table = pa.Table.from_pandas(df, preserve_index=False)
    schema = pa.schema(
        [f.with_type(pa.string()) if pa.types.is_large_string(f.type) else f for f in table.schema]
    )
    return table.cast(schema)


## 2. Escribir Parquet particionado local

La salida queda particionada por `record_type` y `year_month`, una organizacion comun en data lakes.

In [7]:
bronze_files = []
for path in sorted(BRONZE_DIR.glob("*_tripdata_2026-*.parquet")):
    parsed = parse_tlc_filename(path.name)
    if parsed is not None:
        bronze_files.append((path, parsed["record_type"], parsed["year_month"]))

bronze_files[:5], len(bronze_files)


([(PosixPath('/Volumes/Data/flavioc/Codes/GitHub/cursos/Python/diplo-ia/etl-04/data/tlc/raw/fhv_tripdata_2026-01.parquet'),
   'fhv',
   '2026-01'),
  (PosixPath('/Volumes/Data/flavioc/Codes/GitHub/cursos/Python/diplo-ia/etl-04/data/tlc/raw/fhvhv_tripdata_2026-01.parquet'),
   'fhvhv',
   '2026-01'),
  (PosixPath('/Volumes/Data/flavioc/Codes/GitHub/cursos/Python/diplo-ia/etl-04/data/tlc/raw/green_tripdata_2026-01.parquet'),
   'green',
   '2026-01'),
  (PosixPath('/Volumes/Data/flavioc/Codes/GitHub/cursos/Python/diplo-ia/etl-04/data/tlc/raw/yellow_tripdata_2026-01.parquet'),
   'yellow',
   '2026-01')],
 4)

In [8]:
summary = []

for path, record_type, year_month in bronze_files:
    partition_dir = OUTPUT_DIR / f"record_type={record_type}" / f"year_month={year_month}"
    output_path = partition_dir / f"{path.stem}_clean.parquet"
    partition_dir.mkdir(parents=True, exist_ok=True)

    for old_file in partition_dir.glob("*.parquet"):
        old_file.unlink()

    print(f"Processing {path.name} -> {output_path} ...")
    rows_in = 0
    rows_out = 0
    writer = None
    for batch_df in read_parquet_in_batches(path):
        rows_in += len(batch_df)
        clean = transform_batch(batch_df, record_type, year_month, path.name)
        if len(clean):
            table = to_arrow(clean)
            if writer is None:
                writer = pq.ParquetWriter(output_path, table.schema)
            writer.write_table(table)
            rows_out += len(clean)

    if writer is None:
        empty = pd.DataFrame(columns=OUTPUT_COLUMNS)
        empty_table = pa.Table.from_pandas(empty, preserve_index=False)
        pq.write_table(empty_table, output_path)
    else:
        writer.close()
    summary.append(
        {
            "source_file": path.name,
            "record_type": record_type,
            "year_month": year_month,
            "rows_in": rows_in,
            "rows_out": rows_out,
            "rows_removed": rows_in - rows_out,
            "output_path": str(output_path),
        }
    )


Processing fhv_tripdata_2026-01.parquet -> /Volumes/Data/flavioc/Codes/GitHub/cursos/Python/diplo-ia/etl-04/data/tlc/processed/tlc_2026_clean/record_type=fhv/year_month=2026-01/fhv_tripdata_2026-01_clean.parquet ...
Processing fhvhv_tripdata_2026-01.parquet -> /Volumes/Data/flavioc/Codes/GitHub/cursos/Python/diplo-ia/etl-04/data/tlc/processed/tlc_2026_clean/record_type=fhvhv/year_month=2026-01/fhvhv_tripdata_2026-01_clean.parquet ...
Processing green_tripdata_2026-01.parquet -> /Volumes/Data/flavioc/Codes/GitHub/cursos/Python/diplo-ia/etl-04/data/tlc/processed/tlc_2026_clean/record_type=green/year_month=2026-01/green_tripdata_2026-01_clean.parquet ...
Processing yellow_tripdata_2026-01.parquet -> /Volumes/Data/flavioc/Codes/GitHub/cursos/Python/diplo-ia/etl-04/data/tlc/processed/tlc_2026_clean/record_type=yellow/year_month=2026-01/yellow_tripdata_2026-01_clean.parquet ...


In [9]:

summary_df = pd.DataFrame(summary)
summary_df.to_csv(SILVER_DIR / "tlc_2026_transform_summary.csv", index=False)
display(summary_df)


,source_file,record_type,year_month,rows_in,rows_out,rows_removed,output_path
0,fhv_tripdata_2026-01.parquet,fhv,2026-01,1941722,1941607,115,/Volumes/Data/flavioc/Codes/GitHub/cursos/Pyth...
1,fhvhv_tripdata_2026-01.parquet,fhvhv,2026-01,20940373,20940373,0,/Volumes/Data/flavioc/Codes/GitHub/cursos/Pyth...
2,green_tripdata_2026-01.parquet,green,2026-01,40272,40130,142,/Volumes/Data/flavioc/Codes/GitHub/cursos/Pyth...
3,yellow_tripdata_2026-01.parquet,yellow,2026-01,3724889,3684865,40024,/Volumes/Data/flavioc/Codes/GitHub/cursos/Pyth...


## Nota: por qué las carpetas tienen `record_type=...` y `year_month=...`

La salida se guarda con una convención muy común en Data Lakes:

```text
tlc_2026_clean/
  record_type=yellow/
    year_month=2026-01/
      yellow_tripdata_2026-01_clean.parquet
```

A esta forma se la suele llamar **particionado estilo Hive**. No significa que estemos usando Hive; es solo una convención de nombres de carpetas.

La idea es que las carpetas también guardan información. En el ejemplo anterior, el archivo pertenece a:

- `record_type = yellow`
- `year_month = 2026-01`

Esto ayuda a que herramientas como PyArrow, DuckDB, Spark o Athena lean solo las carpetas necesarias. Si consultamos únicamente viajes `yellow` de `2026-01`, no hace falta recorrer todos los archivos del dataset.

Ventaja práctica: menos lectura, consultas más rápidas y menor costo cuando los archivos viven en almacenamiento tipo S3/R2.


## 3. Verificacion de la salida

Leemos el dataset particionado como una sola tabla logica.

In [10]:
partition_schema = pa.schema(
    [
        ("record_type", pa.string()),
        ("year_month", pa.string()),
    ]
)
dataset = ds.dataset(
    OUTPUT_DIR,
    format="parquet",
    partitioning=ds.partitioning(partition_schema, flavor="hive"),
)
clean_preview = dataset.head(10).to_pandas()
display(clean_preview)

partition_counts = dataset.to_table(columns=["record_type", "year_month"]).to_pandas()
display(partition_counts.value_counts(["record_type", "year_month"]).reset_index(name="rows"))


,record_type,year_month,source_file,pickup_datetime,dropoff_datetime,duration_min,pickup_location_id,dropoff_location_id,passenger_count,trip_distance,...,total_amount,base_passenger_fare,tips,driver_pay,cbd_congestion_fee,dispatching_base_num,originating_base_num,affiliated_base_number,shared_request_flag,shared_match_flag
0,fhv,2026-01,fhv_tripdata_2026-01.parquet,2026-01-01 00:02:38,2026-01-01 00:33:13,30.583333,<NA>,14,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,B00112,<NA>,B00112,<NA>,<NA>
1,fhv,2026-01,fhv_tripdata_2026-01.parquet,2026-01-01 00:46:51,2026-01-01 00:54:02,7.183333,<NA>,14,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,B00112,<NA>,B00112,<NA>,<NA>
2,fhv,2026-01,fhv_tripdata_2026-01.parquet,2026-01-01 00:46:13,2026-01-01 01:13:56,27.716667,<NA>,14,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,B00112,<NA>,B00112,<NA>,<NA>
3,fhv,2026-01,fhv_tripdata_2026-01.parquet,2026-01-01 00:17:44,2026-01-01 00:24:31,6.783333,<NA>,14,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,B00112,<NA>,B00112,<NA>,<NA>
4,fhv,2026-01,fhv_tripdata_2026-01.parquet,2026-01-01 00:35:25,2026-01-01 00:40:26,5.016667,<NA>,14,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,B00112,<NA>,B00112,<NA>,<NA>
5,fhv,2026-01,fhv_tripdata_2026-01.parquet,2026-01-01 00:46:07,2026-01-01 00:53:22,7.250000,<NA>,14,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,B00112,<NA>,B00112,<NA>,<NA>
6,fhv,2026-01,fhv_tripdata_2026-01.parquet,2026-01-01 00:23:58,2026-01-01 00:29:38,5.666667,<NA>,14,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,B00112,<NA>,B00112,<NA>,<NA>
7,fhv,2026-01,fhv_tripdata_2026-01.parquet,2026-01-01 00:35:02,2026-01-01 00:37:28,2.433333,<NA>,14,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,B00112,<NA>,B00112,<NA>,<NA>
8,fhv,2026-01,fhv_tripdata_2026-01.parquet,2026-01-01 00:42:21,2026-01-01 00:46:43,4.366667,<NA>,14,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,B00112,<NA>,B00112,<NA>,<NA>
9,fhv,2026-01,fhv_tripdata_2026-01.parquet,2026-01-01 00:56:56,2026-01-01 00:59:57,3.016667,<NA>,14,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,B00112,<NA>,B00112,<NA>,<NA>


,record_type,year_month,rows
0,fhvhv,2026-01,20940373
1,yellow,2026-01,3684865
2,fhv,2026-01,1941607
3,green,2026-01,40130


## 4. Load a Cloudflare R2

Usamos variables de entorno para no dejar credenciales escritas en el notebook. Las definimos en un archivo `.env` en la raíz del repo (que está en `.gitignore`) y las cargamos con `python-dotenv`.

Variables esperadas:

- `R2_ENDPOINT_URL`: endpoint S3-compatible de Cloudflare, por ejemplo `https://<account_id>.r2.cloudflarestorage.com` (sin el nombre del bucket al final).
- `R2_ACCESS_KEY_ID`
- `R2_SECRET_ACCESS_KEY`
- `R2_BUCKET`
- `R2_PREFIX`: carpeta destino opcional, por ejemplo `etl-04/tlc_2026_clean`.

Ejemplo de `.env`:

```text
R2_ENDPOINT_URL=https://<account_id>.r2.cloudflarestorage.com
R2_ACCESS_KEY_ID=...
R2_SECRET_ACCESS_KEY=...
R2_BUCKET=diplo-ai
R2_PREFIX=etl-04/tlc_2026_clean
```

In [ ]:
from dotenv import load_dotenv

# Carga las variables del archivo .env (en la raiz del repo) al entorno del proceso.
# No pisa variables que ya esten definidas en el shell.
load_dotenv(BASE_DIR.parent / ".env")

R2_ENDPOINT_URL = os.environ.get("R2_ENDPOINT_URL")
R2_ACCESS_KEY_ID = os.environ.get("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = os.environ.get("R2_SECRET_ACCESS_KEY")
R2_BUCKET = os.environ.get("R2_BUCKET", "diplo-ai")
R2_PREFIX = os.environ.get("R2_PREFIX", "etl-04/tlc_2026_clean")

missing = [
    name
    for name, value in {
        "R2_ENDPOINT_URL": R2_ENDPOINT_URL,
        "R2_ACCESS_KEY_ID": R2_ACCESS_KEY_ID,
        "R2_SECRET_ACCESS_KEY": R2_SECRET_ACCESS_KEY,
    }.items()
    if not value
]

if missing:
    print("Faltan variables de entorno para subir a R2:", ", ".join(missing))
    print("La transformacion local ya quedo lista en:", OUTPUT_DIR)
else:
    fs = s3fs.S3FileSystem(
        key=R2_ACCESS_KEY_ID,
        secret=R2_SECRET_ACCESS_KEY,
        client_kwargs={"endpoint_url": R2_ENDPOINT_URL},
    )

    uploaded = []
    for local_path in OUTPUT_DIR.rglob("*.parquet"):
        relative_path = local_path.relative_to(OUTPUT_DIR)
        remote_path = f"{R2_BUCKET}/{R2_PREFIX}/{relative_path.as_posix()}"
        fs.put(str(local_path), remote_path)
        uploaded.append(remote_path)

    print(f"Archivos subidos: {len(uploaded)}")
    for path in uploaded[:10]:
        print("s3://" + path)


Resultado esperado:

- Dataset local: `etl-04/data/silver/tlc/tlc_2026_clean/record_type=.../year_month=.../*_clean.parquet`.
- Resumen: `etl-04/data/silver/tlc/tlc_2026_transform_summary.csv`.
- Copia en R2 bajo `s3://<bucket>/<prefix>/...`.